<a href="https://colab.research.google.com/github/Anthea05/Anthea_machine-learning/blob/main/JS03/JS03_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

RANDOM_STATE = 42

In [10]:
url = "https://raw.githubusercontent.com/pkmklong/Breast-Cancer-Wisconsin-Diagnostic-DataSet/master/data.csv"
df = pd.read_csv(url)

print("Shape awal:", df.shape)
df.head()

Shape awal: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [11]:
kolom_tidak_dipakai = ["id", "Unnamed: 32"]
kolom_tidak_dipakai = [c for c in kolom_tidak_dipakai if c in df.columns]
print("Kolom yang TIDAK dipakai:", kolom_tidak_dipakai)

df = df.drop(columns=kolom_tidak_dipakai)

target_col = "diagnosis"
fitur_cols = [c for c in df.columns if c != target_col]
print("Jumlah kolom yang DIPAKAI sebagai fitur:", len(fitur_cols))

X = df[fitur_cols].copy()
y_raw = df[target_col].copy()

Kolom yang TIDAK dipakai: ['id', 'Unnamed: 32']
Jumlah kolom yang DIPAKAI sebagai fitur: 30


In [12]:
le = LabelEncoder()
y = le.fit_transform(y_raw)  # B -> 0, M -> 1

print("Mapping label:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Distribusi kelas:", pd.Series(y).value_counts().to_dict())

Mapping label: {'B': np.int64(0), 'M': np.int64(1)}
Distribusi kelas: {0: 357, 1: 212}


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (455, 30)  Test: (114, 30)


In [14]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif)),
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

In [15]:
k_range = list(range(2, len(fitur_cols) + 1))
param_grid = {"selector__k": k_range}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipeline, param_grid, scoring="accuracy", cv=cv, n_jobs=-1)
grid.fit(X_train, y_train)

best_k = grid.best_params_["selector__k"]
best_cv_score = grid.best_score_
print(f"Jumlah fitur (k) terbaik: {best_k}")
print(f"Rata-rata akurasi cross-validation pada k terbaik: {best_cv_score:.4f}")

Jumlah fitur (k) terbaik: 29
Rata-rata akurasi cross-validation pada k terbaik: 0.9736


In [16]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy  : 0.9649
Precision : 0.9750
Recall    : 0.9286
F1-score  : 0.9512

Confusion matrix:
 [[71  1]
 [ 3 39]]

Classification report:
               precision    recall  f1-score   support

           B       0.96      0.99      0.97        72
           M       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [17]:
selector = best_model.named_steps["selector"]
mask = selector.get_support()
selected_features = np.array(fitur_cols)[mask]
scores = selector.scores_[mask]

hasil_fitur = pd.DataFrame({
    "fitur": selected_features,
    "skor_ANOVA_F": scores
}).sort_values("skor_ANOVA_F", ascending=False).reset_index(drop=True)

print(f"{best_k} fitur terbaik (SelectKBest, f_classif):")
hasil_fitur

29 fitur terbaik (SelectKBest, f_classif):


,fitur,skor_ANOVA_F
0,concave points_worst,733.724933
1,perimeter_worst,717.246487
2,radius_worst,692.861395
3,concave points_mean,684.526845
4,perimeter_mean,548.413236
5,area_worst,522.188947
6,radius_mean,511.274848
7,area_mean,444.857518
8,concavity_mean,397.592082
9,concavity_worst,319.507776


In [18]:
cv_results = pd.DataFrame(grid.cv_results_)[["param_selector__k", "mean_test_score", "std_test_score"]]
cv_results.columns = ["k", "mean_cv_accuracy", "std_cv_accuracy"]
cv_results = cv_results.sort_values("k").reset_index(drop=True)
cv_results

,k,mean_cv_accuracy,std_cv_accuracy
0,2,0.940659,0.027451
1,3,0.945055,0.021978
2,4,0.940659,0.026556
3,5,0.942857,0.028146
4,6,0.947253,0.032894
5,7,0.949451,0.035165
6,8,0.949451,0.035165
7,9,0.947253,0.035710
8,10,0.953846,0.032151
9,11,0.947253,0.027274


Berdasarkan hasil analisa saya, berapa jumlah fitur terbaik yang dapat digunakan? Apa saja fitur tersebut?
Dari hasil pengujian menggunakan SelectKBest dengan fungsi skor ANOVA F-test (f_classif), yang nilai k-nya dicari melalui GridSearchCV dengan 5-fold stratified cross-validation, diperoleh jumlah fitur optimal sebanyak 29 fitur dari total 30 fitur numerik yang tersedia, dengan rata-rata akurasi cross-validation sebesar 97.36%. Namun nilai akurasinya sudah mencapai titik jenuh di angka sama sejak k = 24 fitur. K = 24 sudah dipertimbangkan sebagai jumlah fitur yang efesien karena mampu mempertahankan akurasi yang sama